<a href="https://colab.research.google.com/github/rkfussell/Physics_Affinity/blob/main/physicsAff_priorcoursework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
import re
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
login()

In [5]:
!ls "/content/drive/My Drive/Colab Notebooks"

'Copy of KSG AffinityCoding.ipynb'   physicsAff_useful.ipynb
 physicsAff_priorcoursework.ipynb


In [6]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


confidence = pd.read_excel("/content/drive/My Drive/Colab Notebooks/2024_fall_Cornell_Phys1101and2207_physaff_Rnd2_CollectiveCoding.xlsx",sheet_name="Confidence")
interest = pd.read_excel("/content/drive/My Drive/Colab Notebooks/2024_fall_Cornell_Phys1101and2207_physaff_Rnd2_CollectiveCoding.xlsx",sheet_name="Interest & Use")
confidence_codebook = pd.read_excel("/content/drive/My Drive/Colab Notebooks/2024_fall_Cornell_Phys1101and2207_physaff_Rnd2_CollectiveCoding.xlsx",sheet_name="Codebook-Confidence")
interest_codebook = pd.read_excel("/content/drive/My Drive/Colab Notebooks/2024_fall_Cornell_Phys1101and2207_physaff_Rnd2_CollectiveCoding.xlsx",sheet_name="Codebook-InterestandUse")

Mounted at /content/drive


In [7]:
confidence.head()

,Unnamed: 0,course,studentID,In a few (short) sentences: How confident are you now in\nyour ability to learn physics? What contributes to this?,Confident,Somewhat Confident,Not Confident,Prior Coursework,No Prior Coursework,Time since last class,math confident,Not math confident,skill transfer,effort,inadequacy,covid,(other),Unnamed: 17,Unnamed: 18,Unnamed: 19
0,NaN,PHYS 2207,85ef67d0-649f-4e26-9545-1382bd4086a3,I have never taken a physics class in college ...,NaN,x,NaN,NaN,x,NaN,x,NaN,x,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,PHYS 2207,2d8769ee-10f4-4635-8a2e-f8e6af40abd3,I am pretty confident in my abilities to learn...,x,NaN,NaN,x,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,PHYS 2207,b6b37f0b-dad5-4d8c-acbe-4a72a693eb6e,Physics is pretty hard for me to grasp however...,NaN,NaN,NaN,NaN,NaN,NaN,x,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,PHYS 2207,f8f33e36-9c73-4007-a43b-3fcf1122f2ab,I am not confident in my ability to learn phys...,NaN,NaN,x,NaN,NaN,NaN,NaN,x,x,NaN,x,NaN,NaN,NaN,NaN,NaN
4,NaN,PHYS 2207,dbb358dd-c4e8-4714-84d9-a32e477edfd8,not confident. My AP Physics teacher in high s...,NaN,NaN,x,x,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:

confidence = confidence.rename(columns={"In a few (short) sentences: How confident are you now in\nyour ability to learn physics? What contributes to this?":"questionStem"})
confidence = confidence.fillna(0)
confidence = confidence.replace('x',1)

/tmp/ipykernel_13099/239468420.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  confidence = confidence.replace('x',1)


In [8]:
confidence

,Unnamed: 0,course,studentID,In a few (short) sentences: How confident are you now in\nyour ability to learn physics? What contributes to this?,Confident,Somewhat Confident,Not Confident,Prior Coursework,No Prior Coursework,Time since last class,math confident,Not math confident,skill transfer,effort,inadequacy,covid,(other),Unnamed: 17,Unnamed: 18,Unnamed: 19
0,NaN,PHYS 2207,85ef67d0-649f-4e26-9545-1382bd4086a3,I have never taken a physics class in college ...,NaN,x,NaN,NaN,x,NaN,x,NaN,x,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,PHYS 2207,2d8769ee-10f4-4635-8a2e-f8e6af40abd3,I am pretty confident in my abilities to learn...,x,NaN,NaN,x,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,PHYS 2207,b6b37f0b-dad5-4d8c-acbe-4a72a693eb6e,Physics is pretty hard for me to grasp however...,NaN,NaN,NaN,NaN,NaN,NaN,x,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,PHYS 2207,f8f33e36-9c73-4007-a43b-3fcf1122f2ab,I am not confident in my ability to learn phys...,NaN,NaN,x,NaN,NaN,NaN,NaN,x,x,NaN,x,NaN,NaN,NaN,NaN,NaN
4,NaN,PHYS 2207,dbb358dd-c4e8-4714-84d9-a32e477edfd8,not confident. My AP Physics teacher in high s...,NaN,NaN,x,x,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,NaN,PHYS 2207,d919df7f-ae2e-486f-b563-8d7780fbc994,I don't feel super confident since I tend to a...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
668,NaN,PHYS 2207,b4e7cbfc-8d41-4e31-b841-89b8dbf1d0ba,I am somewhat confident because I took AP phys...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
669,NaN,PHYS 1101,35f501b0-2ac2-4666-bd69-aa71af516978,"Fairly. I believe if I put on enough time, I c...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
670,NaN,PHYS 1101,882ec822-d244-482f-9fea-b2abba2f1643,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
#load model
model_name = "google/gemma-3-4b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    output_hidden_states=True,
    dtype=torch.float32,
    device_map="auto"
)

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

In [37]:

prompt = "Help me analyze some student statements. In each statement, decide if the student says that they do have **Prior Coursework** in physics, that they have **No Prior Coursework** in physics, or you can't tell. \n"
prompt+= "Here is the codebook section for **Prior Coursework** - talking about their prior experiences in physics courses (can be pos or neg). This includes if students describe past experiences as affected by covid, in high school, or even just vaguely 'a long time ago' (for most, we suspect that indicates HS). It does not include self-study (i.e. for the MCAT) or last physics in middle school if stated explicitly. Code does not apply to MCAT prep courses or explicitly partially completed physics courses (i.e. 'I withdrew'). \n"
prompt+= "Here is the codebook section for **No Prior Coursework** - talking about never having taken a physics course before; including last physics class in Middle School, if the student expresses 'no' or 'no proper' background, or if they describe self-study (i.e. studying for the MCAT). /n"
prompt += "Here are some examples:"

prompt+= "\n Example 1: 'To be completely honest, I am worried about my ability to perform well in this course because I did not learn appropriately in high school physics (due to COVID).' \n"

prompt+= "Answer: **Prior Coursework** \n Explanation: The student says 'I did not learn appropriatelyin high school physics' which indicates that they have Prior Coursework in physics."

prompt+= "\n Example 2: 'I am mildly confident in my ability to learn physics. I have never taken a formal physics class before but taught myself the elementary physics equations and concepts for the MCAT.' \n"

prompt+= "Answer: **No Prior Coursework** \n Explanation: The student says 'I have never taken a formal physics course before' which indicates that the student has No Prior Coursework'"

prompt+= "\n Here is another student statement. Tell me if the student says that they have had **Prior Coursework**, **No Prior Coursework**, or **Can't Tell**. Provide your answer first and then explain your reasoning. \n"


In [38]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Run inference
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=3)

In [39]:
# Decode the generated response
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print("Model response:")
print(response)

Model response:
Help me analyze some student statements. In each statement, decide if the student says that they do have **Prior Coursework** in physics, that they have **No Prior Coursework** in physics, or you can't tell. 
Here is the codebook section for **Prior Coursework** - talking about their prior experiences in physics courses (can be pos or neg). This includes if students describe past experiences as affected by covid, in high school, or even just vaguely 'a long time ago' (for most, we suspect that indicates HS). It does not include self-study (i.e. for the MCAT) or last physics in middle school if stated explicitly. Code does not apply to MCAT prep courses or explicitly partially completed physics courses (i.e. 'I withdrew'). 
Here is the codebook section for **No Prior Coursework** - talking about never having taken a physics course before; including last physics class in Middle School, if the student expresses 'no' or 'no proper' background, or if they describe self-study

In [45]:
pcoursework = confidence.loc[confidence['Prior Coursework'] == 1]
npcoursework = confidence.loc[confidence['No Prior Coursework'] == 1]
nocode = confidence.loc[(confidence['Prior Coursework'] == 0) & (confidence['No Prior Coursework'] == 0)]

In [46]:
promptWithStatement=prompt + "``" + pcoursework.iloc[3]['questionStem'] + "''"
print(promptWithStatement)

Help me analyze some student statements. In each statement, decide if the student says that they do have **Prior Coursework** in physics, that they have **No Prior Coursework** in physics, or you can't tell. 
Here is the codebook section for **Prior Coursework** - talking about their prior experiences in physics courses (can be pos or neg). This includes if students describe past experiences as affected by covid, in high school, or even just vaguely 'a long time ago' (for most, we suspect that indicates HS). It does not include self-study (i.e. for the MCAT) or last physics in middle school if stated explicitly. Code does not apply to MCAT prep courses or explicitly partially completed physics courses (i.e. 'I withdrew'). 
Here is the codebook section for **No Prior Coursework** - talking about never having taken a physics course before; including last physics class in Middle School, if the student expresses 'no' or 'no proper' background, or if they describe self-study (i.e. studying 

In [48]:
len(nocode)

604

In [51]:
base=prompt
dTA = nocode

count_pcourse, count_npcourse, count_noCode, count_fuckinGemma =0,0,0,0
with torch.no_grad():
  for i in range(0,20):

      promptWithStatement=base + "``" + dTA.iloc[i]['questionStem'] + "''"
      inputs = tokenizer(promptWithStatement, return_tensors="pt").to(model.device)
      # Run inference
      outputs = model.generate(**inputs, max_new_tokens=10)
      # Decode the generated response
      ans = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
      # Define the first bit to get just Gemma's categorization
      ans_split = ans.split(dTA.iloc[i]['questionStem'])[-1]
      gemmasFeelings = re.findall(r'\*\*(.*?)\*\*',ans_split)

      print("\n")
      print(i)
      # print("The full output is:")
      # print(ans)
      print("The sentence was: ")
      print(dTA.iloc[i]['questionStem'])

      if len(gemmasFeelings) == 0:
          count_fuckinGemma+=1
          print("Gemma feels that it is: ")
          print(ans_split)
          continue
      else:
          print("The stripped answer is: ")
          print(gemmasFeelings[0])

      # print("Gemma's feelings 1st element: ")
      # print(gemmasFeelings[0])

      if "Prior Coursework" == gemmasFeelings[0]:
          count_pcourse+=1
      elif "No Prior Coursework" == gemmasFeelings[0]:
          count_npcourse+=1
      elif "Can't Tell" == gemmasFeelings[0]:
          count_noCode+=1
      else:
          print("The sentence was: ")
          print(dTA.iloc[i]['questionStem'])
          print("Gemma feels that it is: ")
          print(ans_split)

print('\n')
print("Correct: " + str(count_pcourse))
print("Incorrect: " + str(count_npcourse))
print("Can't Tell:" + str(count_noCode))
print("Gemma fucked up:" + str(count_fuckinGemma))



0
The sentence was: 
Physics is pretty hard for me to grasp however I enjoy math and understand that it's a big part of physics. I think I'll have trouble grasping the concepts rather than the math involved. 
The stripped answer is: 
Can't Tell


1
The sentence was: 
I am not confident in my ability to learn physics, because I have never been very good at solving complex math problems and visualizing situations. I also feel discouraged when I see students in class who are able to succeed without putting in much work, because it makes me feel like I should be succeeding at the same rate of them when in reality that is not the case. 
The stripped answer is: 
No Prior Coursework


2
The sentence was: 
I'm not too confident but I'm willing to try my best to learn. I think it's my will to learn that contributes to this.
The stripped answer is: 
Can't Tell


3
The sentence was: 
My honest answer is that I don't know how to feel about my ability to learn physics because it's been a few year